[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/03-OCRfacturas.ipynb)


# 03 — Parsing y Extraction: comparando métodos de OCR

En **`02-PDF_reporte`** vimos que **parsing** (`pypdf`) solo sirve si el PDF ya trae texto seleccionable. En cuanto el "documento" es en realidad una foto, `extract_text()` regresa (casi) nada — la información sigue ahí, pero como píxeles, no como caracteres.

Aquí vemos las dos partes del problema sobre la misma factura:

- **Parsing** (rondas 1-3): imagen → texto plano. Como no hay capa de texto, hace falta OCR en vez de `pypdf`.
- **Extraction** (rondas 4-5, y un ejemplo con regex): texto/imagen → campos específicos (folio, fecha, emisor, total), ya en JSON.

Comparamos **a simple vista**: corres cada ronda, ves lo que imprime, y lo comparas contra la factura real (ábrela junto al notebook).

| Ronda | Método | Qué hace |
|---|---|---|
| 1 | **Tesseract** | Parsing: OCR "clásico", sin deep learning. |
| 2 | **EasyOCR** | Parsing: OCR con redes neuronales. |
| 3 | **PaddleOCR** | Parsing: OCR moderno, más robusto con layouts distintos. |
| 4 | **[Falcon-OCR](https://huggingface.co/tiiuae/Falcon-OCR)** | Extraction: modelo de Hugging Face especializado en documentos — corre local, pero necesita GPU. |
| 5 | **[LlamaExtract](https://cloud.llamaindex.ai)** | Extraction: servicio en la nube, le das el esquema de campos que quieres y te regresa JSON. Sin GPU, pero necesitas tu propia cuenta (gratis). |

Usamos las mismas 3 facturas de ejemplo (inventadas, no son datos reales) en las cinco rondas.


## Carpeta `public/`

Igual que en `02-PDF_reporte`, este notebook espera una carpeta **`public/`** junto al notebook.

Las 3 facturas (`factura_1.pdf`, `factura_2.pdf`, `factura_3.pdf`) son ejemplos inventados para la clase — mismo formato que un CFDI mexicano real, pero con nombres, RFC y montos ficticios. Ya vienen en el repo.

**En Colab:** crea la carpeta `public` (panel de archivos → clic derecho → *Nueva carpeta*) y arrastra ahí `factura_1.pdf`, `factura_2.pdf`, `factura_3.pdf`, `ey_100_casos_rentables_ia_2026.pdf` y `fakeine.jpeg` (descárgalos de `modulo_4/public/` en GitHub).

**En local:** si clonaste el repo, `modulo_4/public/` ya trae los 5 archivos.


In [ ]:
%pip install -q pypdfium2

import time

import pypdfium2 as pdfium
from PIL import Image

FACTURAS = ["factura_1.pdf", "factura_2.pdf", "factura_3.pdf"]


def pdf_a_imagen(ruta_pdf):
    return pdfium.PdfDocument(ruta_pdf)[0].render(scale=2).to_pil().convert("RGB")


## Lo que dice cada factura (el "dato real")

Ábrelas junto al notebook (`public/factura_1.pdf`, etc.) y compáralas contra lo que imprima cada ronda:

| Factura | Folio | Fecha | Emisor | Total |
|---|---|---|---|---|
| `factura_1.pdf` | 001 | 2026-03-14 | Laura Ximena Reyes Cortés | $ 15,660.00 |
| `factura_2.pdf` | 002 | 2026-05-02 | Soluciones Digitales del Bajío | $ 9,512.00 |
| `factura_3.pdf` | 003 | 2026-07-21 | Miguel Ángel Torres Domínguez | $ 25,520.00 |


## Ronda 1 — Tesseract

**Ventajas:** gratis, instantáneo, no necesita GPU.

**Desventajas:** solo texto plano (tú tienes que encontrar el dato); le cuesta con fotos de mala calidad.


In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
%pip install -q pytesseract


In [ ]:
import pytesseract

for nombre in FACTURAS:
    t0 = time.time()
    texto = pytesseract.image_to_string(pdf_a_imagen(f"public/{nombre}"))
    print(f"--- {nombre} ({time.time() - t0:.1f}s) ---")
    print(texto)
    print()


## De parsing a extraction: un ejemplo

El texto de arriba (parsing) es correcto, pero es un bloque de texto -- tú tienes que encontrar el total ahí adentro. Extraer significa ir un paso más allá: pasar de "aquí está todo el texto" a "el total es $15,660.00". Una línea de regex sobre el texto de factura_1.pdf alcanza para hacerlo a mano:


In [ ]:
import re

texto_factura_1 = pytesseract.image_to_string(pdf_a_imagen("public/factura_1.pdf"))
coincidencias = re.findall(r"[TtOo]otal:?\s*\$?\s*([\d.,]+)", texto_factura_1)
print("Total encontrado:", coincidencias[-1] if coincidencias else "no encontrado")


(Buscamos "total" y tomamos la última coincidencia porque "Subtotal" también contiene la palabra "total".) Esto ya da una idea de por qué conviene un método que regrese el dato directo, sin regex -- eso es lo que hacen las rondas 4 y 5.


## Ronda 2 — EasyOCR

**Ventajas:** mejor que Tesseract con fotos "reales" (ángulos, fondos).

**Desventajas:** más lento, descarga un modelo de ~500 MB la primera vez.


In [ ]:
%pip install -q easyocr


In [ ]:
import numpy as np
import easyocr

lector_easyocr = easyocr.Reader(["es"], gpu=False)

for nombre in FACTURAS:
    t0 = time.time()
    lineas = lector_easyocr.readtext(np.array(pdf_a_imagen(f"public/{nombre}")), detail=0)
    print(f"--- {nombre} ({time.time() - t0:.1f}s) ---")
    print("\n".join(lineas))
    print()


## Ronda 3 — PaddleOCR

**Ventajas:** más robusto con layouts distintos que EasyOCR.

**Desventajas:** instalación más pesada; sigue regresando texto plano.


In [ ]:
%pip install -q paddlepaddle paddleocr


In [ ]:
from paddleocr import PaddleOCR

# Si tu version de paddleocr ya no acepta estos parametros, revisa su documentacion actual.
lector_paddle = PaddleOCR(use_angle_cls=True, lang="es")

for nombre in FACTURAS:
    t0 = time.time()
    resultado = lector_paddle.ocr(np.array(pdf_a_imagen(f"public/{nombre}")), cls=True)
    texto = "\n".join(linea[1][0] for bloque in resultado for linea in bloque)
    print(f"--- {nombre} ({time.time() - t0:.1f}s) ---")
    print(texto)
    print()


## Ronda 4 -- Falcon-OCR, un modelo de Hugging Face

Falcon-OCR (de TII) es un modelo chico (300M parametros) especializado en documentos: en vez de texto plano, le pides directo texto, fórmula (LaTeX) o tabla (HTML).

**Necesita GPU.** Antes de seguir: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU T4.

**Ventajas:** entiende tablas sin regex; modelo chico.

**Desventajas:** necesita GPU; solo tiene esos 3 modos fijos, no es conversacional.

Este es el código tal cual viene en la ficha del modelo en Hugging Face (https://huggingface.co/tiiuae/Falcon-OCR):


In [ ]:
%pip install -q -U transformers accelerate


In [ ]:
import torch
from transformers import AutoModelForCausalLM

modelo_falcon = AutoModelForCausalLM.from_pretrained(
    "tiiuae/Falcon-OCR",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

for nombre in FACTURAS:
    t0 = time.time()
    texto = modelo_falcon.generate(pdf_a_imagen(f"public/{nombre}"))[0]
    print(f"--- {nombre} ({time.time() - t0:.1f}s) ---")
    print(texto)
    print()


### El truco: pedirle la tabla directo

Con category="table" te regresa la tabla de renglones ya en HTML. Probemos con factura_3.pdf (la más difícil de las 3 -- mala composición a propósito):


In [ ]:
from IPython.display import HTML, display

tabla_html = modelo_falcon.generate(pdf_a_imagen("public/factura_3.pdf"), category="table")[0]
display(HTML(tabla_html))


## Ronda 5 -- LlamaExtract (servicio en la nube, sin GPU)

LlamaExtract (de la startup LlamaIndex) es distinto a todo lo anterior: no corre en tu computadora ni en Colab, corre en la nube. Le defines los campos que quieres (un esquema) y te regresa JSON.

Necesitas una cuenta gratis en https://cloud.llamaindex.ai (plan gratis: 10,000 usos al mes) y tu propia API key -- nunca la escribas directo en el código, aquí te la va a pedir la celda.

**Ventajas:** no necesita GPU; JSON ya tipado, sin regex.

**Desventajas:** depende de internet y de una cuenta externa; cada quien necesita su propia API key.


In [ ]:
%pip install -q "llama-cloud>=2.8"


In [ ]:
import getpass

from pydantic import BaseModel
from llama_cloud import LlamaCloud

api_key = getpass.getpass("Tu API key de LlamaCloud: ")
cliente_llama = LlamaCloud(api_key=api_key)


class Factura(BaseModel):
    folio: str
    fecha: str
    emisor: str
    total: float


In [ ]:
for nombre in FACTURAS:
    archivo = cliente_llama.files.create(file=f"public/{nombre}", purpose="extract")
    trabajo = cliente_llama.extract.create(
        file_input=archivo.id,
        configuration={"data_schema": Factura.model_json_schema(), "extraction_target": "per_doc"},
    )
    while trabajo.status not in ("COMPLETED", "FAILED", "CANCELLED"):
        time.sleep(2)
        trabajo = cliente_llama.extract.get(trabajo.id)
    print(f"--- {nombre} ---")
    print(trabajo.extract_result)
    print()


### Importa como capturaste el documento: PDF limpio vs. foto del PDF

En la vida real, muchas veces el punto de partida es un PDF y alguien lo imprime y le toma una foto con el celular en vez de mandar el archivo digital. Probamos Falcon-OCR sobre la misma página, capturada de dos formas.


In [ ]:
import numpy as np
from io import BytesIO

pagina_limpia = pdf_a_imagen("public/ey_100_casos_rentables_ia_2026.pdf")

# Simulamos una foto de celular: rotada, mas chica, con sombra y compresion agresiva.
pagina_foto = pagina_limpia.rotate(6, expand=True, fillcolor=(255, 255, 255))
pagina_foto = pagina_foto.resize((pagina_foto.width // 2, pagina_foto.height // 2))
gradiente = np.tile(np.linspace(0.65, 1.0, pagina_foto.width), (pagina_foto.height, 1))
arreglo = np.array(pagina_foto).astype(float)
for canal in range(3):
    arreglo[:, :, canal] *= gradiente
buffer = BytesIO()
Image.fromarray(np.clip(arreglo, 0, 255).astype("uint8")).save(buffer, format="JPEG", quality=40)
buffer.seek(0)
pagina_foto = Image.open(buffer).convert("RGB")

print("--- PDF renderizado limpio ---")
print(modelo_falcon.generate(pagina_limpia)[0][:400])
print()
print("--- Foto simulada del PDF ---")
print(modelo_falcon.generate(pagina_foto)[0][:400])


Si la segunda transcripción tiene más errores o le faltan pedazos, esa es la lección: el modelo importa, pero la calidad de la captura del documento importa tanto o más.


## Cierre

Corriste las mismas 3 facturas por 5 métodos. Ahora compara tú mismo, a ojo (con la tabla de "lo que dice cada factura" de arriba, o abriendo los PDF directamente):

- ¿A cuál método le costó más el folio, la fecha o el total?
- ¿Cuál te dio el dato ya limpio (JSON, en las rondas 4 y 5) y a cuál le tuviste que sacar tú el dato del texto (rondas 1 a 3)?
- ¿Cuál tardó más? ¿Cuál necesitó GPU o internet y cuál no?

No hay una respuesta única de "cuál es el mejor" -- depende de si tienes GPU, si te preocupa mandar documentos a la nube, cuántas facturas vas a procesar, y qué tan complicado es el layout.

En producción, muchas empresas ya resuelven esto con **servicios administrados** (Google Document AI, AWS Textract, Azure Document Intelligence, o el mismo LlamaExtract) que ya vienen entrenados para facturas -- la ventaja de hacerlo "a mano" aquí es entender qué está pasando por dentro antes de delegarlo a una caja negra.


## Bonus opcional -- otro tipo de documento: una identificación

(Sáltate esta sección si vas corto de tiempo.)

Reutilizamos Falcon-OCR sobre una credencial de ejemplo, generada para fines didácticos (public/fakeine.jpeg) -- no es un documento real.


In [ ]:
texto_ine = modelo_falcon.generate(Image.open("public/fakeine.jpeg"))[0]
print(texto_ine)


Aquí no hay una plantilla fija como en las facturas. Con Tesseract, EasyOCR o PaddleOCR tocaria escribir reglas nuevas para nombre, CURP, fecha de nacimiento. Falcon-OCR y LlamaExtract generalizan a otro tipo de documento sin que tengas que escribir nada nuevo.
